# Baseline Holt-Winters (GCP Spark on YARN)
- Input: dense 30-minute demand (from feature engineering)
- Train/val/test: use split column from HDFS
- Model: Holt-Winters per zone (seasonal=48 bins/day)
- Metrics: RMSE, MAE, MAPE, sMAPE, R2
- Output: zone-level predictions and metrics to HDFS

In [1]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, LongType
from statsmodels.tsa.holtwinters import ExponentialSmoothing

BASE_HDFS = "/user/tiennd3886"
DENSE_PATH = f"{BASE_HDFS}/feature_engineering/demand_prediction_dense_30m"
OUT_PRED = f"{BASE_HDFS}/results/baseline_holt_winters_gcp/predictions"
OUT_METRICS = f"{BASE_HDFS}/results/baseline_holt_winters_gcp/metrics"

BIN_COL = "pickup_bin_30m"
ZONE_COL = "PULocationID"
VALUE_COL = "pickup_demand"
SEASONAL_PERIODS = 48

spark = (
    SparkSession.builder
    .appName("BaselineHoltWinters_GCP")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.eventLog.enabled", "true")
    .config("spark.executor.instances", "3")
    .config("spark.executor.cores", "3")
    .config("spark.executor.memory", "8g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.driver.memory", "3g")
    .config("spark.driver.memoryOverhead", "1g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.parquet.mergeSchema", "true")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")  # DISABLE Arrow backend to avoid ChunkedArray
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark ready.")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 15:50:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/25 15:50:11 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Spark ready.


In [2]:
dense = (
    spark.read.parquet(DENSE_PATH)
    .select("split", ZONE_COL, BIN_COL, VALUE_COL)
)
print("Dense rows:", dense.count())


Dense rows: 27684169


In [3]:
# Repartition by zone so each executor gets all rows of a zone
dense_repartitioned = dense.repartition(265, ZONE_COL)

pred_schema = StructType([
    StructField("split", StringType(), True),
    StructField(ZONE_COL, IntegerType(), True),
    StructField(BIN_COL, TimestampType(), True),
    StructField("y_true", DoubleType(), True),
    StructField("y_pred", DoubleType(), True),
])

def forecast_zone(pdf: pd.DataFrame) -> pd.DataFrame:
    """Pure per-zone Holt-Winters forecast. Runs entirely in Pandas; Arrow backend disabled."""
    pdf = pdf.sort_values(BIN_COL).reset_index(drop=True)
    if pdf.empty:
        return pd.DataFrame(columns=["split", ZONE_COL, BIN_COL, "y_true", "y_pred"])

    zone_id = int(pdf[ZONE_COL].iloc[0])
    train = pdf[pdf["split"] == "train"].reset_index(drop=True)
    val   = pdf[pdf["split"] == "val"].reset_index(drop=True)
    test  = pdf[pdf["split"] == "test"].reset_index(drop=True)

    y_train = train[VALUE_COL].astype(float).values
    fallback = float(y_train[-1]) if len(y_train) > 0 else 0.0

    def safe_hw(n):
        if n <= 0:
            return np.zeros(0, dtype=np.float64)
        if len(y_train) < SEASONAL_PERIODS * 2:
            return np.full(n, fallback, dtype=np.float64)
        try:
            m = ExponentialSmoothing(
                y_train, trend="add", seasonal="add",
                seasonal_periods=SEASONAL_PERIODS,
            ).fit(optimized=True)
            fc = m.forecast(n)
            return np.ascontiguousarray(fc, dtype=np.float64)
        except Exception:
            return np.full(n, fallback, dtype=np.float64)

    rows = []
    for subset, preds in [(val, safe_hw(len(val))), (test, safe_hw(len(test)))]:
        for i in range(len(subset)):
            rows.append((
                str(subset["split"].iat[i]),
                zone_id,
                subset[BIN_COL].iat[i],
                float(subset[VALUE_COL].iat[i]),
                float(preds[i]),
            ))

    if not rows:
        return pd.DataFrame(columns=["split", ZONE_COL, BIN_COL, "y_true", "y_pred"])

    return pd.DataFrame(rows, columns=["split", ZONE_COL, BIN_COL, "y_true", "y_pred"])


predictions = (
    dense_repartitioned
    .groupBy(ZONE_COL)
    .applyInPandas(forecast_zone, schema=pred_schema)
    .cache()
)
print("Pred rows:", predictions.count())


Pred rows: 8482802


In [4]:
metrics_base = (
    predictions
    .withColumn("err", F.col("y_true") - F.col("y_pred"))
    .withColumn("abs_err", F.abs(F.col("err")))
    .withColumn("sq_err", F.col("err") ** 2)
    .withColumn("abs_pct", F.when(F.col("y_true") != 0, F.abs(F.col("err") / F.col("y_true"))).otherwise(F.lit(None)))
    .withColumn("smape", F.when(
        (F.abs(F.col("y_true")) + F.abs(F.col("y_pred"))) != 0,
        2 * F.abs(F.col("err")) / (F.abs(F.col("y_true")) + F.abs(F.col("y_pred")))
    ).otherwise(F.lit(None)))
)

agg = (
    metrics_base.groupBy("split")
    .agg(
        F.sqrt(F.avg("sq_err")).alias("rmse"),
        F.avg("abs_err").alias("mae"),
        F.avg("abs_pct").alias("mape"),
        F.avg("smape").alias("smape"),
        F.avg("y_true").alias("y_mean"),
    )
)

sse = metrics_base.groupBy("split").agg(F.sum("sq_err").alias("sse"))
sst = (
    metrics_base
    .join(agg.select("split", "y_mean"), on="split", how="left")
    .withColumn("sst", (F.col("y_true") - F.col("y_mean")) ** 2)
    .groupBy("split")
    .agg(F.sum("sst").alias("sst"))
)

metrics = (
    agg.join(sse, on="split", how="left")
    .join(sst, on="split", how="left")
    .withColumn("r2", F.when(F.col("sst") != 0, 1 - (F.col("sse") / F.col("sst"))).otherwise(F.lit(None)))
)
metrics.show(truncate=False)

predictions.write.mode("overwrite").parquet(OUT_PRED)
metrics.write.mode("overwrite").parquet(OUT_METRICS)
print("Saved to:", OUT_PRED)
print("Saved to:", OUT_METRICS)

spark.catalog.clearCache()
spark.stop()


+-----+-----------------+------------------+-----------------+------------------+-----------------+---------------------+-------------------+------------------+
|split|rmse             |mae               |mape             |smape             |y_mean           |sse                  |sst                |r2                |
+-----+-----------------+------------------+-----------------+------------------+-----------------+---------------------+-------------------+------------------+
|val  |63.88813691940227|11.121678522590495|2.385597100401627|1.651845634327843 |8.863156567973933|1.1075150236383087E10|2.003615954133588E9|-4.527581377825597|
|test |127.0799353563598|20.054268427397133|2.825727853480054|1.6692113438839251|10.37729682528485|9.317232957054585E10 |5.240164376723951E9|-16.78042116090934|
+-----+-----------------+------------------+-----------------+------------------+-----------------+---------------------+-------------------+------------------+



Saved to: /user/tiennd3886/results/baseline_holt_winters_gcp/predictions
Saved to: /user/tiennd3886/results/baseline_holt_winters_gcp/metrics
